# GradCAM — Analyse de l'attention de l'EarEncoder

Ce notebook explore **où** le réseau regarde quand il encode une oreille en embedding.

| Expérience | Question |  
|---|---|  
| 1 — Heatmap moyenne | Quelles zones anatomiques sont universellement importantes ? |  
| 2 — Comparaison sujets | L'attention varie-t-elle selon la morphologie de l'oreille ? |  
| 3 — Gauche vs droite | Y a-t-il une asymétrie attentionnelle gauche/droite ? |  
| 4 — Évolution entraînement | Comment l'attention se spécialise-t-elle au fil des epochs ? |

---
**Architecture** : EfficientNetB0 backbone (7×7×1280) → GlobalAveragePooling2D → MLP → embedding L2-normalisé.  
Grad-CAM calcule ∂score/∂feat_maps puis pondère les 1280 cartes de features.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Chargement du dataset et du modèle

In [ ]:
from src.multimodal import MultimodalDataset

DATASET_PATH = "../dataset/preprocessed_dataset.npz"

dataset = MultimodalDataset.load(DATASET_PATH)
print(f"Dataset chargé : {dataset.n_subjects} sujets")
print(f"Sujets disponibles : {dataset.subject_ids}")
print(f"Images L : {dataset._img_L.shape}  (N, H, W, C), valeurs [0, 1]")
print(f"HRTF     : {dataset._hrtf.shape}")

In [ ]:
import numpy as np
from src.models import EarEncoder, HRTFEncoder, ContrastiveModel

WEIGHTS_PATH = "../checkpoints/phase1_best.weights.h5"

ear_encoder  = EarEncoder(embedding_dim=128)
n_sh, n_freqs, _ = dataset.hrtf_shape
hrtf_encoder = HRTFEncoder(n_sh=n_sh, n_freqs=n_freqs, embedding_dim=128)
contrastive  = ContrastiveModel(ear_encoder, hrtf_encoder, temperature=0.07)

dummy = {
    "ear_left":  np.zeros((1, 224, 224, 3), dtype="float32"),
    "ear_right": np.zeros((1, 224, 224, 3), dtype="float32"),
    "hrtf":      np.zeros((1, n_sh, n_freqs, 2), dtype="float32"),
}
_ = contrastive(dummy, training=False)

contrastive.load_weights(WEIGHTS_PATH)
print("Poids chargés.")
ear_encoder.model.summary()


In [ ]:
from src.explainability import GradCAM

# Instanciation construit les deux spy models en interne
gradcam = GradCAM(ear_encoder, hrtf_encoder)
print("GradCAM prêt.")

---
## Expérience 1 — Heatmap consensus

On moyenne les heatmaps sur tous les sujets pour identifier les zones **universellement importantes**.

In [ ]:
from src.explainability import experiment_average_heatmap

# Score A : norme L2 de l'embedding (aucune référence HRTF nécessaire)
avg_hm = experiment_average_heatmap(
    gradcam     = gradcam,
    dataset     = dataset,
    subject_ids = dataset.subject_ids,   # tous les sujets
    ear         = "left",
    score_fn    = None,                  # score_norm par défaut
    save_path   = "../results/gradcam_exp1_consensus.png",
)
print(f"Max heatmap consensus : {avg_hm.max():.3f}  |  Moyenne : {avg_hm.mean():.3f}")

In [ ]:
# Score B : similarité avec l'embedding HRTF d'un sujet de référence
# quels pixels prédisent CE profil HRTF précis ?
#plus précisement : quels pixels de l'oreille font que le réseau prédit un profil HRTF similaire à celui du sujet X ?
#Objectif est de répondre à ces question :
# "Quels détails anatomiques font que H10 ressemble à H11 ?" → Score B avec hrtf_emb de H11
#"Quels pixels séparent H10 des autres ?" → Score B avec le hrtf_emb moyen de tous les autres sujets
#"Est-ce que le réseau regarde le même endroit pour deux sujets avec des HRTF similaires ?"

ref_subject_id = 1
ref_subject = dataset.subject_ids[ref_subject_id]

_, _, hrtf_ref = dataset._img_L[ref_subject_id], dataset._img_R[ref_subject_id], dataset._hrtf[ref_subject_id]
hrtf_emb = hrtf_encoder.model.predict(hrtf_ref[np.newaxis], verbose=0)

score_sim = GradCAM.score_similarity(hrtf_emb)

avg_hm_sim = experiment_average_heatmap(
    gradcam     = gradcam,
    dataset     = dataset,
    subject_ids = dataset.subject_ids,
    ear         = "left",
    score_fn    = score_sim,
    save_path   = "../results/gradcam_exp1_consensus_sim.png",
)

---
## Expérience 2 — Comparaison inter-sujets

L'attention est-elle la même pour des oreilles morphologiquement différentes ?

In [ ]:
from src.explainability import experiment_compare_subjects

# Choisir 4 sujets contrastés (adapte les IDs à ton dataset)
selected = dataset.subject_ids[:4]

heatmaps = experiment_compare_subjects(
    gradcam     = gradcam,
    dataset     = dataset,
    subject_ids = selected,
    ear         = "left",
    score_fn    = None,
    save_path   = "../results/gradcam_exp2_inter_sujets.png",
)

# Corrélation entre heatmaps (diversité attentionnelle)
print("\nCorrélation de Pearson entre heatmaps :")
flat = {k: v.flatten() for k, v in heatmaps.items()}
ids  = list(flat.keys())
corr = np.corrcoef([flat[i] for i in ids])
fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(corr, vmin=0, vmax=1, cmap="Greens")
ax.set_xticks(range(len(ids))); ax.set_xticklabels(ids, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(len(ids))); ax.set_yticklabels(ids, fontsize=7)
plt.colorbar(im, ax=ax, label='corrélation')
ax.set_title("Similarité des heatmaps", fontsize=9)
plt.tight_layout(); plt.show()

---
## Expérience 3 — Asymétrie gauche / droite

Le réseau encode-t-il les deux oreilles de la même façon ?

In [ ]:
from src.explainability import experiment_left_vs_right

subject_id = dataset.subject_ids[2]  

hm_L, hm_R = experiment_left_vs_right(
    gradcam    = gradcam,
    dataset    = dataset,
    subject_id = subject_id,
    score_fn   = None,
    save_path  = f"../results/gradcam_exp3_LR_{subject_id}.png",
)

diff = np.abs(hm_L - hm_R)
print(f"Différence L/R — max : {diff.max():.3f}  |  moyenne : {diff.mean():.3f}")
print(f"Corrélation  L↔R : {np.corrcoef(hm_L.flatten(), hm_R.flatten())[0,1]:.3f}")

---
## Expérience 4 — Évolution au cours de l'entraînement

On charge des checkpoints à différentes époques pour voir comment l'attention se spécialise.

In [ ]:
#Lance un entrainement qui sauvegarde les checkpoints dans checkpoints

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import tensorflow as tf
from src.multimodal import MultimodalDataset
from src.models import EarEncoder, HRTFEncoder, ContrastiveModel
from src.models.trainer import Trainer

# ── 1. Dataset ────────────────────────────────────────────────────────────────
DATASET_PATH = "../dataset/preprocessed_dataset.npz"
dataset = MultimodalDataset.load(DATASET_PATH)

# ── 2. Modèle ─────────────────────────────────────────────────────────────────
ear_encoder  = EarEncoder(embedding_dim=128)
hrtf_encoder = HRTFEncoder(n_sh=121, n_freqs=129, embedding_dim=128)
model        = ContrastiveModel(ear_encoder, hrtf_encoder, temperature=0.07, supervised=True)

dummy = {
    "ear_left":  np.zeros((1, 224, 224, 3), dtype="float32"),
    "ear_right": np.zeros((1, 224, 224, 3), dtype="float32"),
    "hrtf":      np.zeros((1, 121, 129, 2), dtype="float32"),
}
model(dummy, training=False)

# ── 3. Callback epoch checkpoint ─────────────────────────────────────────────
EPOCHS_TO_SAVE = [1, 2, 3, 4, 5]
CKPT_DIR       = "../checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

class EarEncoderEpochCheckpoint(tf.keras.callbacks.Callback):
    def __init__(self, ear_enc, save_dir, epochs_to_save):
        super().__init__()
        self._ear_enc        = ear_enc
        self._save_dir       = save_dir
        self._epochs_to_save = set(epochs_to_save)

    def on_epoch_end(self, epoch, logs=None):
        ep = epoch + 1   # Keras indexe à partir de 0
        if ep in self._epochs_to_save:
            path = os.path.join(self._save_dir, f"epoch_{ep:02d}.weights.h5")
            self._ear_enc.model.save_weights(path)
            print(f"\n  [EpochCkpt] epoch {ep:2d} sauvegardée → {path}")

epoch_ckpt = EarEncoderEpochCheckpoint(ear_encoder, CKPT_DIR, EPOCHS_TO_SAVE)

# ── 4. Entraînement via Trainer ───────────────────────────────────────────────
trainer = Trainer(
    model,
    ear_encoder,
    experiment   = "gradcam_evolution",
    dataset_path = DATASET_PATH,
    base_dir     = "../checkpoints",
    dataset      = dataset,
)

trainer.run_phase(
    1,
    dataset.train,
    dataset.val,
    epochs          = max(EPOCHS_TO_SAVE),   # 20  EarlyStopping peut arrêter avant
    extra_callbacks = [epoch_ckpt],
)

trainer.end()
trainer.plot_history()

# ── 5. Vérification ───────────────────────────────────────────────────────────
print("\nCheckpoints disponibles :")
for ep in EPOCHS_TO_SAVE:
    p = os.path.join(CKPT_DIR, f"epoch_{ep:02d}.weights.h5")
    print(f"  epoch {ep:2d} → {'✓' if os.path.exists(p) else '✗ MANQUANT'} — {p}")


In [ ]:
from src.models import EarEncoder
from src.explainability import experiment_training_evolution

# Checkpoints enregistrés pendant l'entraînement
# Adapte les chemins à tes fichiers
CHECKPOINT_PATHS = [
    "../checkpoints/epoch_01.weights.h5",
    "../checkpoints/epoch_02.weights.h5",
    "../checkpoints/epoch_03.weights.h5",
    "../checkpoints/epoch_04.weights.h5",
    "../checkpoints/epoch_05.weights.h5",
]
LABELS = ["Époque 1", "Époque 2", "Époque 3", "Époque 4", "Époque 5"]

encoders = []
for path in CHECKPOINT_PATHS:
    enc = EarEncoder(embedding_dim=128)
    enc.model.load_weights(path)
    encoders.append(enc)

subject_id = dataset.subject_ids[0]

heatmaps_evo = experiment_training_evolution(
    ear_encoders  = encoders,
    dataset       = dataset,
    subject_id    = subject_id,
    ear           = "left",
    epoch_labels  = LABELS,
    save_path     = f"../results/gradcam_exp4_evolution_{subject_id}.png",
)

In [ ]:
# Entropie des heatmaps : une entropie élevée = attention diffuse
#                         une entropie faible  = attention focalisée
def heatmap_entropy(hm: np.ndarray, eps: float = 1e-8) -> float:
    p = hm.flatten() / (hm.sum() + eps)
    return float(-np.sum(p * np.log(p + eps)))

print("Évolution de la focalisation de l'attention :")
for label, hm in zip(LABELS, heatmaps_evo):
    print(f"  {label:12s} → entropie = {heatmap_entropy(hm):.3f}  (↓ = plus focalisé)")

---
## Explorer les dimensions de l'embedding (Score C)

Quels pixels activent une direction spécifique de l'espace latent ?

In [ ]:
subject_id_num = 3
subject_id = dataset.subject_ids[subject_id_num]
img_L, img_R, _ = dataset._img_L[subject_id_num], dataset._img_R[subject_id_num], dataset._hrtf[subject_id_num]

DIMS_TO_EXPLORE = [0, 10, 42, 64, 100, 127]

fig, axes = plt.subplots(2, len(DIMS_TO_EXPLORE), figsize=(3 * len(DIMS_TO_EXPLORE), 6))
fig.suptitle(f"Dimensions de l'embedding — sujet {subject_id}", fontsize=11, fontweight='bold')

for col, dim in enumerate(DIMS_TO_EXPLORE):
    score_fn = GradCAM.score_dim(dim)
    hm = gradcam.compute(img_L, img_R, ear='left', score_fn=score_fn)
    
    axes[0, col].imshow(hm, cmap='jet', vmin=0, vmax=1)
    axes[0, col].set_title(f"dim {dim}", fontsize=8)
    axes[0, col].axis('off')
    
    axes[1, col].imshow(np.clip(GradCAM.overlay(img_L, hm), 0, 1))
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Heatmap', fontsize=8)
axes[1, 0].set_ylabel('Superposition', fontsize=8)
plt.tight_layout()
plt.savefig('../results/gradcam_bonus_dimensions.png', dpi=150, bbox_inches='tight')
plt.show()